In [0]:
# Cargar tabla bronce_fincas
bronce_fincas = spark.table("testetlfincas.bronze.bronce_fincas")

# Cargar dimensiones
dim_educacion = spark.table("testetlfincas.silver.dim_educacion")
dim_estado_civil = spark.table("testetlfincas.silver.dim_estadoCivil")
dim_genero = spark.table("testetlfincas.silver.dim_genero")
dim_tenencia = spark.table("testetlfincas.silver.dim_tenencia")
dim_uen = spark.table("testetlfincas.silver.dim_uen")

# Reemplazar columnas de descripción por IDs
fact_fincas = bronce_fincas \
    .join(dim_educacion, bronce_fincas["EDUCACION"] == dim_educacion["EDUCACION"], "left") \
    .join(dim_estado_civil, bronce_fincas["ESTADO_CIVIL"] == dim_estado_civil["ESTADO_CIVIL"], "left") \
    .join(dim_genero, bronce_fincas["GENERO"] == dim_genero["GENERO"], "left") \
    .join(dim_tenencia, bronce_fincas["TENENCIA"] == dim_tenencia["TENENCIA"], "left") \
    .join(dim_uen, bronce_fincas["UEN"] == dim_uen["UEN"], "left") \
    .select(
        dim_educacion["id_Educacion"].alias("id_Educacion"),
        dim_estado_civil["id_EstadoCivil"].alias("id_EstadoCivil"),
        dim_genero["id_Genero"].alias("id_Genero"),
        dim_tenencia["id_Tenencia"].alias("id_Tenencia"),
        dim_uen["id_UEN"].alias("id_UEN"),
        *[col for col in bronce_fincas.columns if col not in ["EDUCACION", "ESTADO_CIVIL", "GENERO", "TENENCIA", "UEN"]]
    )

# Normalizar nombres de columna (reemplazar caracteres inválidos por guiones bajos)
import re
fact_fincas = fact_fincas.toDF(*[re.sub(r'[ ,;{}()\n\t=]', '_', c) for c in fact_fincas.columns])

# Guardar la fact table en el schema Gold
fact_fincas.write.format("delta").mode("overwrite").saveAsTable("testetlfincas.gold.FactFincas")